# Cosmos Reason 2 - End-to-end VLM Compression

Deploying a reasoning Vision-Language Model in production is not just a matter of accuracy — it is a matter of cost, latency, and throughput. [Cosmos Reason 2](https://github.com/nvidia-cosmos/cosmos-reason2) is purpose-built for Physical AI: it reasons over images and video to support robotic manipulation, autonomous navigation, and embodied decision-making. These applications demand real-time or near-real-time responses, often on constrained hardware budgets. A 2 B parameter BF16 model is already compact by frontier standards, but even modest compression — 15 % fewer parameters and FP8 weights — can double serving throughput and halve memory footprint, making the difference between a model that fits on a single GPU and one that requires two. The challenge is doing this without sacrificing the chain-of-thought reasoning quality that makes Cosmos Reason 2 valuable in the first place. This notebook shows that with the right pipeline — structured pruning, knowledge distillation, and PTQ — you can reach that bar.

In this course you will learn how to apply the full compression pipeline — prune → distill → quantize → serve → evaluate — on NVIDIA's [`nvidia/Cosmos-Reason2-2B`](https://huggingface.co/nvidia/Cosmos-Reason2-2B), a reasoning-specialised Vision-Language Model (VLM).

**Pruning** focuses on removing parts of the model that are most redundant, **distillation** aims to recover the quality of the pruned model to match the original and **quantization** reduces the precision of weights, activations and KV cache. In this notebook, you will learn how to apply each technique to Cosmos Reason 2 while preserving the accuracy of the model.

**Compression target.** The compressed checkpoint at the end (pruned LM \~1.4 B + distilled + FP8 — full VLM \~1.7 B) should land within a couple of points of the BF16 baseline on BLINK (PAI-Bench proxy) and RealWorldQA. §6 sets up a 4-way comparison so you can read off `% recovered` for each benchmark. The 15 % cut is on the LM tower (1.7 B → \~1.4 B); the vision encoder + projector (\~0.72 B) is untouched, so the full VLM goes 2.0 B → \~1.7 B. The cut is intentionally conservative — the goal here is *full* recovery.

Pipeline overview:

| Stage | What it does | Wall-time (approx.) |
|---|---|---|
| §1 Download | Pull the gated HF checkpoint (Cosmos-Reason2-2B, ~5 GB; LM tower 1.7 B + vision/projector 0.72 B) | 5 min |
| §2 Prune | Minitron depth-pruning on the LM tower, 28 → 24 layers (~15 % LM cut) | 5–10 min |
| §3 Distill | Text-only KD on the pruned LM tower | 20–30 min |
| §4 Quantize | FP8 PTQ with image-aware calibration | 10–15 min |
| §5 Serve | vLLM OpenAI-compatible endpoint | <2 min to spin up |
| §6 Evaluate | 4-way comparison × BLINK / RealWorldQA | 20–30 min |

---
## 1. Download Cosmos Reason 2 and Inspect the Architecture

> 🔐 Cosmos Reason 2 is **gated** on HuggingFace. Before running §1 request access at the model page and authenticate: `hf auth login --token <YOUR_HF_TOKEN>`.

We start by pulling the checkpoint locally and reading its `config.json`. Then we inspect the architecture of the model.

In [ ]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
from huggingface_hub import login, snapshot_download

# HF_TOKEN is injected by docker run -e HF_TOKEN=<your_token>
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Pass it to docker run with: -e HF_TOKEN=hf_...")
login(token=HF_TOKEN, add_to_git_credential=False)

BASE_PATH = "/workspace"
COSMOS_PATH = f"{BASE_PATH}/Cosmos-Reason2-2B"

snapshot_download(
    repo_id="nvidia/Cosmos-Reason2-2B",
    local_dir=COSMOS_PATH,
    token=HF_TOKEN,
    local_dir_use_symlinks=False,
)
!ls {COSMOS_PATH} | head


We start by inspecting the architecture of [Cosmos Reason 2](https://github.com/nvidia-cosmos/cosmos-reason2). **Cosmos Reason 2** is a reasoning-specialised VLM built on the **Qwen3-VL** family (`Qwen3VLForConditionalGeneration`). It has three main components:

```
Cosmos-Reason2-2B  (2.44 B total)
├── Vision Encoder   (ViT-style, ~0.42 B)   — frozen during pruning/distill
├── Vision Projector (MLP, ~0.30 B)          — frozen during pruning/distill
└── Language Model   (Qwen3, 1.72 B)         ← compression target
```

A standard VLM architecture looks like the following:

![VLM architecture](VLM.png)

**Language Model tower** (`model.language_model`, 1.72 B):

| Hyperparameter | Value |
| --- | --- |
| `num_hidden_layers` | 28 |
| `hidden_size` | 2048 |
| `num_attention_heads` | 16 |
| `num_key_value_heads` | 8 |
| `head_dim` | 128 |
| `intermediate_size` | 6144 |
| `vocab_size` | 151 936 |
| `max_position_embeddings` | 262 144 |

**Vision Encoder** (24-layer ViT):

| Hyperparameter | Value |
| --- | --- |
| `depth` | 24 |
| `hidden_size` | 1024 |
| `intermediate_size` | 4096 |
| `num_heads` | 16 |
| `patch_size` | 16 × 16 px |
| `spatial_merge_size` | 2 |
| `temporal_patch_size` | 2 |
| `out_hidden_size` | 2048 |
| `deepstack_visual_indexes` | [5, 11, 17] |

**Input resolution** (dynamic):
- `min_pixels`: 65 536 (≈ 256 × 256)
- `max_pixels`: 16 777 216 (≈ 4096 × 4096)
- Images are resized so total pixel count falls in `[min_pixels, max_pixels]` while preserving aspect ratio. After the 2×2 spatial merge, each group of 4 patches maps to one visual token.

Note that we are targeting compression primarily of the LLM component given the size compared to the total size of the model, we will try to decrease the size of it. Research also shows that the vision encoders tend to be very sensitive to compression, so we will omit it here. We aim to decrease the number of parameters of the model by about 15%. 

Run the code cell below to load the full VLM, resolve the LM tower path (`model.language_model`), count parameters for each component, and print the config values above.

In [ ]:
from transformers import AutoConfig, AutoModelForImageTextToText, AutoProcessor
from modelopt.torch.utils.plugins.vlm import resolve_vlm_lm_path
from IPython.display import display
from PIL import Image
import requests, torch

cfg = AutoConfig.from_pretrained(COSMOS_PATH, trust_remote_code=True)
tc  = cfg.text_config

vlm = AutoModelForImageTextToText.from_pretrained(
    COSMOS_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(COSMOS_PATH, trust_remote_code=True)

# resolve_vlm_lm_path returns a dotted path like "model.language_model" — the same one
# the prune/distill scripts use to extract / reinsert the LM tower.
lm_path = resolve_vlm_lm_path(vlm)
lm = vlm
for part in lm_path.split("."):
    lm = getattr(lm, part)

total_b = sum(p.numel() for p in vlm.parameters()) / 1e9
lm_b    = sum(p.numel() for p in lm.parameters()) / 1e9
print(f"Total VLM params:               {total_b:.2f} B")
print(f"  Language tower ({lm_path}):  {lm_b:.2f} B")
print(f"  Vision + projector + heads:   {total_b - lm_b:.2f} B")
print()
print(f"architectures:           {cfg.architectures}")
print(f"text_config.model_type:  {tc.model_type}")
print(f"num_hidden_layers:       {tc.num_hidden_layers}")
print(f"hidden_size:             {tc.hidden_size}")
print(f"num_attention_heads:     {tc.num_attention_heads}")
print(f"num_key_value_heads:     {tc.num_key_value_heads}")
print(f"intermediate_size:       {tc.intermediate_size}")

Now let's run a quick sanity-check inference with the **baseline BF16 model** before touching anything. We feed it four images and ask for a one-sentence caption or to answer a question.

We'll rerun these exact prompts at every compression stage (post-prune, post-distill, post-FP8) so you can track caption drift by eye. Full quantitative evaluation — BLINK and RealWorldQA — is deferred to §6 once all stages are complete.

> **Expected outputs**:
>
> | Image | Question | Expected answer |
> |---|---|---|
> | <img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg" width="160"/> | Describe the image in one short sentence. | `"A detailed diagram of a volcano, showcasing its internal structure and external features, including magma movement, volcanic activity, and surrounding geological layers."` |
> | <img src="http://images.cocodataset.org/val2017/000000037777.jpg" width="160"/> | Which is closer to the camera: the bowl of fruit or the stove? | `"The bowl of fruit"` |
> | <img src="http://images.cocodataset.org/val2017/000000039769.jpg" width="160"/> | How many remote controls are in the picture? | `"2"` |
> | <img src="http://images.cocodataset.org/val2017/000000252219.jpg" width="160"/> | Look at the pedestrian crossing signal. Safe to cross? | `"unsafe. The pedestrian crossing signal is showing a red hand, indicating that it is unsafe to cross."` |
>
> Coherent, on-topic answers mean the model loaded correctly. Garbled or off-topic output is a red flag — re-check the download in §1 before proceeding.


In [ ]:
# Shared probe suite — volcano (AI2D sanity check) + three spatially-grounded COCO questions.
# Reused at every compression stage so drift is visible by eye across baseline/prune/distill/FP8.
from PIL import Image
import requests as _req
from IPython.display import display

PROBES = [
    {"label": "volcano (AI2D)",
     "url":   "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg",
     "q":     "Describe the image in one short sentence."},
    {"label": "depth — bowl vs stove",
     "url":   "http://images.cocodataset.org/val2017/000000037777.jpg",
     "q":     "Which is closer to the camera: the bowl of fruit or the stove? Answer briefly."},
    {"label": "counting — remotes",
     "url":   "http://images.cocodataset.org/val2017/000000039769.jpg",
     "q":     "How many remote controls are in the picture? Answer with a single number."},
    {"label": "localization — crossing signal",
     "url":   "http://images.cocodataset.org/val2017/000000252219.jpg",
     "q":     "Look at the pedestrian crossing signal. Is it safe to cross? "
              "Answer 'safe' or 'unsafe', then explain in one sentence."},
]

def run_probes_hf(model, processor):
    "Run all four probe images through model and print results."
    for p in PROBES:
        img = Image.open(_req.get(p["url"], stream=True).raw).convert("RGB")
        thumb = img.copy(); thumb.thumbnail((400, 400)); display(thumb)
        msgs = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text":  p["q"]},
        ]}]
        inputs = processor.apply_chat_template(
            msgs, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        answer = processor.batch_decode(
            out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        print(f"  [{p['label']}] {answer}")


In [ ]:
print("\n=== Baseline BF16 ===")
run_probes_hf(vlm, processor)


In [ ]:
# Empty the memory to proceed
import gc; del vlm; gc.collect(); torch.cuda.empty_cache()

---
## 2. Prune the LLM

In this section we are going to prune the LLM component of Cosmos Reason 2.

For pruning we are going to apply [`prune_minitron.py`](../prune_minitron.py) on the VLM checkpoint. The script auto-detects the VLM, extracts its language tower to a temporary causal-LM dir, runs activation-importance calibration on `nemotron-post-training-dataset-v2`, prunes the inner LM, then reinserts the pruned LM back into the original VLM container at `--output_hf_path`.

`--prune_export_config` selects the target shape directly. For simplicity we are going to apply **depth pruning** by removing 4 layers from the 28-layer model. You can further apply **width pruning** for shrinking the FFN sizes. 

Megatron-Core has no VLM provider, so [`prune_minitron.py`](../prune_minitron.py) handles this by **extracting** the language tower into a plain HF causal LM, running the regular `mcore_minitron` algorithm on it, then **reinserting** the pruned LM back into the original VLM container. The vision encoder, projector and `lm_head` are copied verbatim.

For more details on model pruning techniques and best practices, check out the [Minitron blog](https://developer.nvidia.com/blog/how-to-prune-and-distill-llama-3-1-8b-to-an-nvidia-llama-3-1-minitron-4b-model/) and accompanying [technical report](https://arxiv.org/pdf/2408.11796). Further pruning recipes and best practices can also be found in the [pruning guidelines notes](https://github.com/NVIDIA/Model-Optimizer/tree/main/examples/pruning#pruning-guidelines).

![Minitron pruning pipeline](Pruning.png)

Expect the run to take roughly 5–10 minutes on a single H100.

For more advanced pruning, you can apply Neural Architecture Search (NAS). This way we only specify the target size of the model, and the algorithm will then generate candidates with different pruning configs that satisfy this requirement. The candidates will then be evaluated and the best checkpoint will be saved. If you are interested in running NAS pruning, here is an example set of arguments to prune_minitron.py:

```bash
torchrun --nproc_per_node 2 ../prune_minitron.py \
    --pp_size 2 \
    --hf_model_name_or_path $COSMOS_PATH \
    --prune_target_params 1.6e9 \
    --hparams_to_skip hidden_size num_attention_heads \
    --ss_channel_divisor 256 \    
    --prune_score_func mmlu_10pct_bs32 \
    --calib_num_samples 512 \
    --seq_length 4096 \
    --top_k 10 \
    --output_hf_path $PRUNED_PATH
```

> ⚠️ `--hparams_to_skip hidden_size` is **mandatory** for NAS. Without it NAS will happily pick a smaller `hidden_size` to hit the budget, and the reinsertion step will refuse to splice the smaller LM back into the original vision projector (`ValueError` in `reinsert_pruned_lm_into_vlm`), meaning that the projector will need to be retrained. If you intentionally want a smaller `hidden_size`, drop the flag and plan on retraining the projector separately.

In [ ]:
PRUNED_PATH = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-24"
COSMOS_PATH = f"{BASE_PATH}/Cosmos-Reason2-2B"

!torchrun --nproc_per_node 1 ../prune_minitron.py \
    --pp_size 1 \
    --hf_model_name_or_path {COSMOS_PATH} \
    --prune_export_config '{{"num_layers": 24}}' \
    --calib_num_samples 512 \
    --seq_length 4096 \
    --output_hf_path {PRUNED_PATH}

Now let's verify that the pruned model matches our target shape and regenerate the caption.

The code cell checks three things:
1. **Parameter count** — the LM tower should drop to ~1.4–1.5 B after removing 4 of 28 layers; the vision encoder + projector should be unchanged.
2. **Shape** — We have only updated the `num_hidden_layers`.
3. **Caption** — the pruned model runs the same AI2D prompt as the baseline. Compare the two outputs.

**What to expect.** Don't worry if the model predictions have changed! Pruning removes whole layers without any recovery training, so caption quality typically drops — shorter sentences, less detail, occasionally factual drift. This degradation is intentional and expected: the next stage (§3 knowledge distillation) is specifically designed to recover it by training the pruned student against the unpruned teacher.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from modelopt.torch.utils.plugins.vlm import resolve_vlm_lm_path
from IPython.display import display
from PIL import Image
import requests, torch

pruned = AutoModelForImageTextToText.from_pretrained(
    PRUNED_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(PRUNED_PATH, trust_remote_code=True)
tc = pruned.config.text_config

lm_path = resolve_vlm_lm_path(pruned)
lm = pruned
for part in lm_path.split("."):
    lm = getattr(lm, part)
total_b = sum(p.numel() for p in pruned.parameters()) / 1e9
lm_b    = sum(p.numel() for p in lm.parameters()) / 1e9

print(f"Total VLM params:              {total_b:.2f} B")
print(f"  Language tower ({lm_path}): {lm_b:.2f} B  (pruned)")
print(f"  Vision + projector + heads:  {total_b - lm_b:.2f} B  (unchanged)")
print()
print(f"num_hidden_layers:   {tc.num_hidden_layers} (pruned) ")
print(f"num_attention_heads: {tc.num_attention_heads}  ")
print(f"num_key_value_heads: {tc.num_key_value_heads}  ")
print(f"intermediate_size:   {tc.intermediate_size}  ")
print(f"hidden_size:         {tc.hidden_size}  # locked at original")

print("\n=== Post-prune ===\n")
run_probes_hf(pruned, processor)

import gc; del pruned; gc.collect(); torch.cuda.empty_cache()


Compare the post-prune outputs above with the baseline outputs from §1. Did the answers get worse? Which of the four probes degraded the most, and why might that probe be more sensitive to layer removal than the others?

Minitron removes the 4 layers with the lowest activation-importance scores. Which layers do you think got dropped by the algorithm?

What would happen if we were to remove more layers? How about applying width pruning by reducing the intermediate FFN size?

---
## 3. Knowledge Distillation: Recover Lost Quality

Now we apply text-only knowledge distillation from the original Cosmos Reason 2 (teacher) into the pruned VLM (student), via [`distill.py`](../distill.py) from the [Megatron-Bridge](https://github.com/NVIDIA-NeMo/Megatron-Bridge) framework. The wrapper extracts both LM towers, runs `kd_loss` distillation on text tokens, then reinserts the distilled student LM back into the original VLM container.

### Distillation Theory

Knowledge distillation ([Hinton, 2015](https://arxiv.org/pdf/1503.02531)) trains a smaller **student** model to mimic a larger **teacher** model, recovering the quality lost during pruning without increasing model size. The teacher (original Cosmos Reason 2) supervises the student (pruned model) on a training corpus.

Two main variants exist:

- **Hard distillation**: The teacher generates token labels; the student fine-tunes on these pseudo-labels. Simple, but only transfers the teacher's final decisions.
- **Soft distillation**: The student minimises the KL divergence between its output distribution and the teacher's, preserving the full probability signal including the teacher's uncertainty:

$$
\mathcal{L}_{\text{KD}} = D_{\text{KL}}\!\left(P_{\text{teacher}} \,\|\, P_{\text{student}}\right) = \sum_x P_{\text{teacher}}(x) \log \frac{P_{\text{teacher}}(x)}{P_{\text{student}}(x)}
$$

Soft distillation is more data-efficient and produces better students because it conveys richer information than hard labels — the teacher's near-zero probabilities over wrong tokens still carry a meaningful signal about which tokens are *plausibly* wrong.

**Dataset choice matters.** Cosmos Reason 2 is a *reasoning* VLM, so the distillation corpus must include chain-of-thought content to preserve reasoning ability. We blend Nemotron-CC (general text), Nemotron-SFT-Math, and Nemotron-SFT-Code at weights 0.1 / 0.5 / 0.4 — heavily favouring structured reasoning over plain prose.

### 3.1 Training Dataset Preparation

**Cosmos Reason 2 is a reasoning VLM**, so the distillation corpus should preserve reasoning. We blend three Nemotron sources at sampling weights `0.1 / 0.5 / 0.4`:

| Dataset (HF) | Subset | Weight | Why |
|---|---|---:|---|
| `nvidia/Nemotron-Pretraining-Dataset-sample` | `Nemotron-CC-High-Quality-Synthetic` | 0.1 | General high-quality synthetic text — keeps the base language capability healthy. |
| `nvidia/Nemotron-Pretraining-SFT-v1` | `Nemotron-SFT-Math` | 0.5 | Chain-of-thought math reasoning. |
| `nvidia/Nemotron-Pretraining-SFT-v1` | `Nemotron-SFT-Code` | 0.4 | Chain-of-thought code reasoning. |

Tokenise each separately so they end up as **three distinct `.bin`/`.idx` prefixes**, then pass them to `distill.py` via `--data_paths <w> <prefix> <w> <prefix> <w> <prefix>`. All three are pretraining-style raw-text corpora (`text` key) → use `--append_eod`. See [`../../dataset/MEGATRON_DATA_PREP.md`](../../dataset/MEGATRON_DATA_PREP.md) for chat-formatted alternatives.

Expect 10-15 min for the tokenization process to complete.

In [ ]:
import os, json
from itertools import islice
from datasets import load_dataset

RAW_DIR       = f"{BASE_PATH}/distill_data_cosmos/raw"
TOKENIZED_DIR = f"{BASE_PATH}/distill_data_cosmos/tokenized_cosmos"
os.makedirs(RAW_DIR, exist_ok=True)

# Streaming mode pulls parquet shards on demand, so disk usage tracks MAX_SAMPLES, not
# the source size. At gbs=1024, seq=2048, a 200-300 iter run consumes ~0.4-0.6 B tokens; with 3 shards
# averaging ~500-1000 tokens/doc the 1 M-per-shard cap below leaves enough variety
# that the schedule does not loop. Drop to 200_000 for a quick run on smaller amount of data.
MAX_SAMPLES = 1_000_000

# Each tuple: (hf_dataset, hf_subset, output_jsonl_stem, sampling_weight).
SOURCES = [
    ("nvidia/Nemotron-Pretraining-Dataset-sample", "Nemotron-CC-High-Quality-Synthetic", "nemotron_cc",  0.1),
    ("nvidia/Nemotron-Pretraining-SFT-v1",         "Nemotron-SFT-MATH",                  "nemotron_math", 0.5),
    ("nvidia/Nemotron-Pretraining-SFT-v1",         "Nemotron-SFT-Code",                  "nemotron_code", 0.4),
]

for hf, subset, stem, _ in SOURCES:
    jsonl_path = f"{RAW_DIR}/{stem}.jsonl"
    if os.path.exists(jsonl_path) and os.path.getsize(jsonl_path) > 0:
        print(f"  {stem:<14} (cached) → {jsonl_path}")
        continue
    print(f"  {stem:<14} streaming up to {MAX_SAMPLES:,} docs from {hf}::{subset}...")
    ds = load_dataset(hf, subset, split="train", streaming=True)
    n = 0
    with open(jsonl_path, "w") as f:
        for row in islice(ds, MAX_SAMPLES):
            text = row.get("text", "")
            if not text or not text.strip():
                continue
            f.write(json.dumps({"text": text}) + "\n")
            n += 1
            if n % 10_000 == 0:
                print(f"    ... wrote {n:,}")
    print(f"  {stem:<14} {n:>7} docs → {jsonl_path}")

In [ ]:
# Tokenise all three JSONLs in one shot. megatron_preprocess_data with multiple
# --jsonl_paths produces one .bin/.idx per input, so we get three separately-weightable
# prefixes that we'll plumb into distill.py below.
PRUNED_PATH = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-24"
TOKENIZED_DIR = f"{BASE_PATH}/distill_data_cosmos/tokenized_cosmos"

!python -m modelopt.torch.utils.plugins.megatron_preprocess_data \
    --jsonl_paths \
        {BASE_PATH}/distill_data_cosmos/raw/nemotron_cc.jsonl \
        {BASE_PATH}/distill_data_cosmos/raw/nemotron_math.jsonl \
        {BASE_PATH}/distill_data_cosmos/raw/nemotron_code.jsonl \
    --json_keys text \
    --tokenizer {PRUNED_PATH} \
    --output_dir {TOKENIZED_DIR} \
    --workers 8 \
    --append_eod

!ls {TOKENIZED_DIR}

### 3.2 Knowledge Distillation

Run the next cell to start distillation! 
Expect 20-30 min to complete.

Note that we are only training the model for 50 iterations. Ideally it would need to be trained longer until convergence for better results. You can shorten the number of iterations to get results quicker but expect lower quality than with full training convergence.

Note that we are also omitting validation during training. You can re-enable it with two flags:
- **`--eval_iters N`** — number of batches to run per validation pass (default: 32). Set to `0` to skip entirely.
- **`--eval_interval N`** — how often to run a validation pass, in training steps (default: 100). This also controls checkpoint save frequency.

**Optional:** Monitor training loss with TensorBoard.

Open a terminal in JupyterLab (**File → New → Terminal**) and run:

```bash
tensorboard --logdir /workspace/Cosmos-Reason2-pruned-depth-distill --port 6006 --bind_all
```

Then open [http://localhost:6006](http://localhost:6006) in your browser (the Docker `-p 6006:6006` mapping and your SSH tunnel forward it through). Ignore the URL TensorBoard prints — it shows the container hostname which won't resolve.

Loss values appear after the first checkpoint is written — give it a minute. Press **Ctrl+C** to stop.

In [ ]:
COSMOS_PATH   = f"{BASE_PATH}/Cosmos-Reason2-2B"
PRUNED_PATH   = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-24"
TOKENIZED_DIR = f"{BASE_PATH}/distill_data_cosmos/tokenized_cosmos"
DISTILL_WORK_DIR = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-distill"
DISTILLED_PATH   = f"{BASE_PATH}/Cosmos-Reason2-distilled"

# Sampling weights must sum to 1.0; distill.py mixes them per training step.
# Blend is math-heavy (0.5) + code (0.4) + general CC (0.1) to favour reasoning recovery.
DATA_PATHS = (
    f"0.1 {TOKENIZED_DIR}/nemotron_cc_text "
    f"0.5 {TOKENIZED_DIR}/nemotron_math_text "
    f"0.4 {TOKENIZED_DIR}/nemotron_code_text"
)

!torchrun --nproc_per_node 4 ../distill.py \
    --tp_size 1 \
    --student_hf_path {PRUNED_PATH} \
    --teacher_hf_path {COSMOS_PATH} \
    --data_paths {DATA_PATHS} \
    --data_path_to_cache {TOKENIZED_DIR}/cache \
    --seq_length 2048 --mbs 4 --gbs 1024 \
    --train_iters 50 \
    --lr 3e-5 --min_lr 1e-5 --lr_warmup_iters 0 \
    --log_interval 1 \
    --eval_iters 0 \
    --output_dir {DISTILL_WORK_DIR} \
    --hf_export_path {DISTILLED_PATH} \
    --student_hf_model {PRUNED_PATH}

You should expect to see distillation loss like the following:

<img src="Distillation_loss.png" width="480"/>

The loss should drop steeply in the first few hundred iterations and then plateau. At 50 iters (as run here) you are still on the steep part of the curve — expect partial recovery. For full recovery, train until the curve fully flattens and on a larger dataset.

In [ ]:
# Verify that the distillation has completed and the final checkpoint is saved in the HF format.
!ls "{BASE_PATH}/Cosmos-Reason2-distilled"

Distillation is now complete!
Again, let's verify that the distilled VLM generates coherent answers.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from IPython.display import display
from PIL import Image
import requests, torch

vlm = AutoModelForImageTextToText.from_pretrained(
    DISTILLED_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(DISTILLED_PATH, trust_remote_code=True)

print("\n=== Post-distill ===\n")
run_probes_hf(vlm, processor)

import gc; del vlm; gc.collect(); torch.cuda.empty_cache()


---
## 4. Post-training Quantization (FP8)

Quantization is a powerful technique for compressing deep learning models by reducing the precision of the numerical values used to represent model parameters, activations, and intermediate data. Instead of relying on high-precision formats like FP32 (32-bit floating point), quantization transforms these values into lower-bit representations such as FP8, INT8, NVFP4, INT4, or even down to binary (1-bit). This process unlocks several compelling advantages:
- Dramatic reduction in overall model footprint
- Lower memory bandwidth requirements
- Faster inference, especially on GPUs and resource-constrained edge devices

There are three primary strategies for quantizing models:
- **Post-Training Quantization (PTQ)**: This approach converts a fully trained model to a lower-precision format after training is complete.
- **Quantization-Aware Training (QAT)**: Here quantization is integrated directly into the training process. During both the forward and backward passes, quantization and dequantization operations are simulated, making the model robust to quantization noise that it will encounter during inference.
- **Quantization-Aware Distillation (QAD)**: Extends QAT by training a low-precision student model under the supervision of a high-precision teacher, using a distillation loss tailored to the quantized student. This helps recover near–full-precision accuracy even at very low bit-widths like FP8 and NVFP4.

In your opinion, which method do you think better preserves model accuracy?

In this section, we are going to apply post-training FP8 quantization to the distilled model we obtained earlier.

For PTQ we use [`../../llm_ptq/hf_ptq.py`](../../llm_ptq/hf_ptq.py). 

`--calib_with_images` builds the calibration set from image-text pairs so activation ranges in the projector → LM interface reflect real multimodal traffic — important for a VLM whose forward pass mixes visual and text tokens. NVFP4 is available with `--qformat nvfp4` on Blackwell.

We are going to create FP8 checkpoints for the original model and the distilled one for a full comparison.

In [ ]:
# Original model quantization
FP8_PATH_ORIG = f"{BASE_PATH}/Cosmos-Reason2-fp8"

!python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path "{BASE_PATH}/Cosmos-Reason2-2B" \
    --export_path   {FP8_PATH_ORIG} \
    --qformat fp8 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code

In [ ]:
# Distilled model quantization
FP8_PATH = f"{BASE_PATH}/Cosmos-Reason2-distilled-fp8"

!python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path {DISTILLED_PATH} \
    --export_path   {FP8_PATH} \
    --qformat fp8 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code

In [ ]:
# Verify Quantized model exported Cosmos-Reason2-distilled-fp8
!ls "{BASE_PATH}/Cosmos-Reason2-distilled-fp8"

---
## 5. Serve with vLLM

Now that the compressed models are ready, we can start a [vLLM](https://vllm.ai/) OpenAI-compatible endpoint on the FP8 export. Run this in a separate terminal (or backgrounded), then return here to evaluate.

```bash
export BASE_PATH=/workspace
vllm serve $BASE_PATH/Cosmos-Reason2-distilled-fp8 \
    --trust-remote-code --dtype bfloat16 \
    --mm-processor-kwargs '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'
```

The `--mm-processor-kwargs` cap is important for a reasoning VLM that routinely sees dense diagrams — without it a single high-resolution image can blow the per-request VRAM budget.

In [ ]:
# Poll vLLM until it's ready.
import requests, time
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("vLLM is ready"); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print("vLLM not reachable after 10 min — check the serve logs")

Once the pruned, distilled and quantized model is deployed, we are once again going to verify that it is still able to generate captions for images. How do you expect the result to compare?

> **Important:** Before running the cell below, make sure the vLLM instance you started in §5 is still running. If you stopped it, restart it with the `vllm serve` command from the cell above. After you are done with the inference check, stop the server before moving to §6 — the evaluation driver in §6 manages its own vLLM lifecycle on the same port and will fail if the port is already in use.
>
> ```bash
> pkill -f 'vllm serve'
> ```

In [ ]:
import requests

served_model = requests.get("http://localhost:8000/v1/models").json()["data"][0]["id"]
print(f"served: {served_model}\n")

for p in PROBES:
    resp = requests.post(
        "http://localhost:8000/v1/chat/completions",
        headers={"Authorization": "Bearer token-abc123"},
        timeout=120,
        json={
            "model": served_model,
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": p["url"]}},
                {"type": "text", "text": p["q"]},
            ]}],
            "max_tokens": 64,
            "temperature": 0,
        },
    ).json()
    print(f"  [{p['label']}] {resp['choices'][0]['message']['content']}")


---
## 6. Evaluate — 4-way comparison

In the final section, the goal is to show that the compressed checkpoint (pruned + distilled + FP8) recovers almost all of the baseline 2B BF16 quality. To make the recovery delta legible we evaluate **four checkpoints** on both **quantitative benchmarks** and a **qualitative showdown**, with a single driver that keeps each vLLM server up for both.

| Model Variant | Path | What it tells us |
|---|---|---|
| **baseline** | `nvidia/Cosmos-Reason2-2B` (BF16) | Reference number to recover. |
| **distilled_bf16** | `{DISTILLED_PATH}` (BF16) | What KD recovers before quantization. |
| **distilled_fp8** | `{FP8_PATH}` | Shippable artifact. |
| **orig_fp8** | `{FP8_PATH_ORIG}` | Original model quantized. |

**Quantitative benchmarks** (lmms-eval):

| Benchmark | Why | Task names |
|---|---|---|
| **BLINK · relative-depth** | Which object is nearer? Core to grasping / obstacle avoidance. | `blink_relative_depth` |
| **BLINK · counting** | How many objects? Manipulation / inventory. | `blink_counting` |
| **BLINK · object-localization** | Where is it in the frame? Servoing / navigation. | `blink_object_localization` |
| **RealWorldQA** | General real-world VQA guardrail — catches broad regressions. | `realworldqa` |

**Qualitative showdown**: while each vLLM server is live, the driver also asks three spatially-grounded questions (relative depth, counting, object localization) on real COCO photos — the same skills the BLINK benchmarks test. Results are rendered in the cell below the driver.

Note that we only ran the distilled model for 50 iterations. To ensure full qualitative recovery, please run distillation for longer (200-300 iterations or until convergence on a larger dataset).

We loop **vLLM-up → showdown → lmms-eval → vLLM-down** per variant. Expect ~20-30 min end-to-end on 1 H100.

> Stop the `vllm serve` you started in §5 before running this section — the driver manages its own vLLM lifecycle on the same port.


Now we are ready to run evaluation! The following cells serve each model variant and evaluates them on each benchmark.

Allow 20-30 min for this cell to complete.

In [ ]:
# 4-way driver: vLLM up → qualitative showdown → lmms-eval → vLLM down.
# Results land under EVAL_ROOT/<variant>/{paibench,realworldqa}/ and showdown.json.
import base64, os, subprocess, time, shlex, json, signal
import requests

EVAL_ROOT = f"{BASE_PATH}/logs/cosmos-r2-recovery-eval"
os.makedirs(EVAL_ROOT, exist_ok=True)

VARIANTS = {
    "baseline":       COSMOS_PATH,     # unpruned Cosmos-Reason2-2B (BF16)
    "distilled_bf16": DISTILLED_PATH,  # pruned + distilled, BF16
    "distilled_fp8":  FP8_PATH,        # pruned + distilled + FP8 (final)
    "orig_fp8":       FP8_PATH_ORIG,
}
BENCHMARKS = {
    "realworldqa": ["realworldqa"],
    "paibench":    ["blink_relative_depth", "blink_counting", "blink_object_localization"],
}

# Showdown: one question per BLINK subtask tested above — same skill, real COCO photos.
SHOWDOWN = [
    {"id": "depth",
     "url": "http://images.cocodataset.org/val2017/000000037777.jpg",
     "q": "Which is closer to the camera: the bowl of fruit or the stove? Answer briefly.",
     "truth": "bowl"},
    {"id": "counting",
     "url": "http://images.cocodataset.org/val2017/000000039769.jpg",
     "q": "How many remote controls are in the picture? Answer with a single number.",
     "truth": "2|two"},
    {"id": "localization",
     "url": "http://images.cocodataset.org/val2017/000000252219.jpg",
     "q": "Look at the pedestrian crossing signal in this image. Is it currently safe "
          "to cross? Answer with the single word 'safe' or 'unsafe', then explain in one sentence.",
     "truth": "unsafe"},
]
SHOW_DIR = f"{BASE_PATH}/showdown"
os.makedirs(SHOW_DIR, exist_ok=True)
for item in SHOWDOWN:
    item["file"] = f"{SHOW_DIR}/{item['id']}.jpg"
    if not os.path.exists(item["file"]):
        r = requests.get(item["url"], timeout=30); r.raise_for_status()
        open(item["file"], "wb").write(r.content)

MM_KW = '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'


def serve_vllm(ckpt: str):
    cmd = (
        f"vllm serve {shlex.quote(ckpt)} "
        "--trust-remote-code --dtype bfloat16 "
        f"--mm-processor-kwargs {shlex.quote(MM_KW)}"
    )
    return subprocess.Popen(cmd, shell=True, preexec_fn=os.setsid)


def wait_ready(timeout_s: int = 600):
    for _ in range(timeout_s // 5):
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False


def stop_vllm(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    p.wait(timeout=60)


def ask(model, item):
    b64 = base64.b64encode(open(item["file"], "rb").read()).decode()
    r = requests.post("http://localhost:8000/v1/chat/completions", timeout=180, json={
        "model": model, "max_tokens": 48, "temperature": 0,
        "messages": [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            {"type": "text", "text": item["q"]}]}]})
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


def run_lmms_eval(ckpt: str, tasks: list, out_dir: str):
    if not tasks:
        return
    os.makedirs(out_dir, exist_ok=True)
    env = {**os.environ, "OPENAI_API_KEY": "token-abc123"}
    cmd = [
        "python", "-m", "lmms_eval",
        "--model", "openai",
        "--model_args", f"model={ckpt},base_url=http://localhost:8000/v1",
        "--tasks", ",".join(tasks),
        "--batch_size", "32",
        "--output_path", out_dir,
    ]
    subprocess.run(cmd, env=env, check=False)


answers = {}
for variant, ckpt in VARIANTS.items():
    print(f"\n{'='*60}\n>>> {variant}  ({ckpt})\n{'='*60}")
    proc = serve_vllm(ckpt)
    try:
        if not wait_ready():
            print(f"!! vLLM failed to come up for {variant}; skipping.")
            continue
        # (1) qualitative showdown while the server is live
        answers[variant] = {}
        for item in SHOWDOWN:
            print(f"  showdown: {item['id']}")
            try:
                answers[variant][item["id"]] = ask(ckpt, item)
            except Exception as e:
                answers[variant][item["id"]] = f"<error: {e}>"
        json.dump({"items": SHOWDOWN, "answers": answers},
                  open(f"{EVAL_ROOT}/showdown.json", "w"), indent=2)
        # (2) quantitative lmms-eval
        for bench_name, tasks in BENCHMARKS.items():
            out_dir = f"{EVAL_ROOT}/{variant}/{bench_name}"
            print(f"  -> {bench_name}: {tasks}")
            run_lmms_eval(ckpt, tasks, out_dir)
    finally:
        stop_vllm(proc)

print(f"\nAll variants done. Results under {EVAL_ROOT}/")


#### Qualitative Showdown

Accuracy tables tell you *how often* a variant is right; here we look at the answers visually. While each server was live, the driver asked all four variants the same three spatially-grounded questions matching the BLINK subtasks above:

| Question | Skill tested | BLINK task |
|---|---|---|
| Bowl vs. stove — which is closer? | Relative depth | `blink_relative_depth` |
| How many remote controls? | Counting | `blink_counting` |
| Safe to cross the pedestrian signal? | Object localization + interpretation | `blink_object_localization` |

The ✓/✗ is a word-match heuristic for eyeballing, not a score — read the actual answers.


In [ ]:
# Render the showdown: image + question + all four answers.
import json, re
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import HTML, display

EVAL_ROOT = f"{BASE_PATH}/logs/cosmos-r2-recovery-eval"
S = json.load(open(f"{EVAL_ROOT}/showdown.json"))
ORDER = ["baseline", "distilled_bf16", "distilled_fp8", "orig_fp8"]

def mark(ans, truth):
    return "✅" if any(re.search(rf"\b{re.escape(t)}\b", ans.lower())
                        for t in truth.lower().split("|")) else "❌"

for item in S["items"]:
    fig, ax = plt.subplots(figsize=(4.8, 3.5))
    ax.imshow(mpimg.imread(item["file"])); ax.axis("off")
    ax.set_title(f"[{item['id']}]  {item['q']}", fontsize=9, wrap=True)
    plt.show()
    rows = "".join(
        f"<tr><td style='padding:2px 14px;white-space:nowrap'><b>{v}</b></td>"
        f"<td style='padding:2px 14px;text-align:left'>{S['answers'].get(v, {}).get(item['id'], '<i>not run</i>')}</td>"
        f"<td style='text-align:center'>{mark(S['answers'].get(v, {}).get(item['id'], ''), item['truth'])}</td></tr>"
        for v in ORDER)
    display(HTML(
        "<table style='font-size:13px;border-collapse:collapse'>"
        "<tr><th></th><th style='text-align:left'>answer</th>"
        f"<th>truth: \"{item['truth'].split('|')[0]}\"</th></tr>" + rows + "</table>"))

### Aggregate the comparison

Collect per-variant scores and render the recovery table. `% recovered` is the score expressed as a percentage of the BF16 baseline — 100 % means the variant matches baseline exactly. The shippable artifact is `distilled_fp8`; we want its row to be ≥ 90 % of baseline on every benchmark.

In [ ]:
# Walk EVAL_ROOT/<variant>/<bench>/ for lmms-eval result JSONs and render a table.
import os, json, glob

EVAL_ROOT = f"{BASE_PATH}/logs/cosmos-r2-recovery-eval"
VARIANT_ORDER = ["baseline", "distilled_bf16", "distilled_fp8", "orig_fp8"]

# Each entry: (display_key, bench_dir, task_substring_to_match)
BENCH_COLS = [
    ("B:depth",   "paibench", "blink_relative_depth"),
    ("B:count",   "paibench", "blink_counting"),
    ("B:localiz", "paibench", "blink_object_localization"),
    ("RWQA",      "realworldqa", "realworldqa"),
]


def headline_metric(results_json: dict, task_substring: str):
    if "results" not in results_json:
        return None
    for task_name, metrics in results_json["results"].items():
        if task_substring not in task_name:
            continue
        for k, v in metrics.items():
            if not isinstance(v, (int, float)):
                continue
            key = k.split(",")[0]  # strip lmms-eval's ",none" suffix
            if key.endswith(("_stderr", "_stderr_clt", "_stderr_clustered")):
                continue
            if key.startswith(("exact_match", "acc", "blink_acc")):
                return float(v) * (100 if v <= 1 else 1)
    return None


def latest_result(variant: str, bench_dir: str):
    # lmms-eval writes <timestamp>_results.json — match *results*.json to catch both orderings
    candidates = sorted(glob.glob(f"{EVAL_ROOT}/{variant}/{bench_dir}/**/*results*.json", recursive=True))
    if not candidates:
        return None
    with open(candidates[-1]) as f:
        return json.load(f)


def pct(num):
    return f"{num:5.1f}" if isinstance(num, float) else "  ?  "


scores = {v: {} for v in VARIANT_ORDER}
for v in VARIANT_ORDER:
    for col, bench_dir, task_sub in BENCH_COLS:
        rj = latest_result(v, bench_dir)
        scores[v][col] = headline_metric(rj, task_sub) if rj else None

col_names = [c[0] for c in BENCH_COLS]
print(f"{'variant':<18} | " + " | ".join(f"{c:>10}" for c in col_names))
print("-" * 70)
for v in VARIANT_ORDER:
    row = " | ".join(pct(scores[v][c]) for c in col_names)
    print(f"{v:<18} | {row}")

print("\n% recovered (vs. baseline):")
print(f"{'variant':<18} | " + " | ".join(f"{c:>10}" for c in col_names))
print("-" * 70)
for v in VARIANT_ORDER:
    cells = []
    for col in col_names:
        base = scores["baseline"].get(col)
        cur = scores[v].get(col)
        if None in (base, cur) or base == 0:
            cells.append("     -    ")
        else:
            cells.append(f"{cur / base * 100:>8.1f}%  ")
    print(f"{v:<18} | " + " | ".join(cells))

---
## 7. Inference Performance Benchmark

Accuracy is half the story. Compression earns its place only if the smaller / lower-precision model **serves faster in the target environment**. For example, a typical robotics use-case may run its *own* perception→reasoning loop: one camera frame in, one decision out, **one or few requests at a time**, not a datacenter queue of hundreds. So we benchmark the regime that actually ships: **batch-1 plus a small fan-out (concurrency 1 → 8)** for the multi-camera case.

We run [aiperf](https://github.com/ai-dynamo/aiperf) against a live vLLM endpoint with a **representative robotics VLM workload (one 768 × 768 image + a short instruction)** and read four relevant metrics:

| Metric | What it means on a robot | Better |
|---|---|---|
| **End-to-end latency / decision** | Wall-clock for the full answer — the number a control loop budgets against. | lower |
| **TTFT** (time-to-first-token) | Reaction time — delay before the first decision token. For a VLM this is **vision-tower + image-prefill bound**. | lower |
| **Per-stream decode rate** (tokens/s/stream) | How fast the model streams its reasoning once started. **LM-decode bound** — this is where quantization + pruning from previous steps pay off. | higher |
| **Aggregate throughput** (tokens/s) | *Secondary* — only matters if one GPU fans out to several cameras / robots. | higher |

**Two settings make the comparison fair:**

1. **`--extra-inputs ignore_eos:true`** forces every variant to generate the *full* 100-token budget. Without it, a lightly-distilled student that stops early generates fewer tokens, and its decode-rate / throughput look *worse* purely because it produced less work — an artifact, not a slowdown.
2. **Time-based steady state** (`--warmup-duration` + `--benchmark-duration`) so kernel autotuning / CUDA-graph capture is warm and the measured window is stable.

> 🔥 Stop the manual `vllm serve` from §5 first — the driver below manages its own vLLM lifecycle on port 8000.

In [ ]:
import os, glob, subprocess, time, shlex, signal
import requests

BENCH_ROOT = f"{BASE_PATH}/benchmarks/cosmos-r2-aiperf"
os.makedirs(BENCH_ROOT, exist_ok=True)

VARIANTS = {
    "baseline":       COSMOS_PATH,
    "orig_fp8":       FP8_PATH_ORIG,
    "distilled_bf16": DISTILLED_PATH,
    "distilled_fp8":  FP8_PATH,
}
CONCURRENCIES = [1, 2, 4, 8]
IMG_W, IMG_H  = 768, 768
TEXT_ISL      = 80
OSL           = 100

MM_KW = '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'


def serve_vllm(ckpt, log_path):
    cmd = (f"vllm serve {shlex.quote(ckpt)} --trust-remote-code --dtype bfloat16 "
           f"--mm-processor-kwargs {shlex.quote(MM_KW)}")
    return subprocess.Popen(cmd, shell=True, stdout=open(log_path, "w"),
                            stderr=subprocess.STDOUT, preexec_fn=os.setsid)


def wait_ready(timeout_s=900):
    for _ in range(timeout_s // 5):
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False


def stop_vllm(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        p.wait(timeout=60)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except ProcessLookupError: pass


def run_aiperf(ckpt, concurrency, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    if glob.glob(f"{out_dir}/**/profile_export_aiperf.json", recursive=True):
        print(f"    c={concurrency} already done — skipping"); return
    cmd = [
        "aiperf", "profile",
        "--model", ckpt,
        "--url", "http://localhost:8000",
        "--endpoint-type", "chat",
        "--streaming",
        "--concurrency", str(concurrency),
        "--warmup-duration", "10",
        "--benchmark-duration", "30",
        "--benchmark-grace-period", "15",
        "--image-batch-size", "1",
        "--image-width-mean",  str(IMG_W),
        "--image-height-mean", str(IMG_H),
        "--image-width-stddev", "0",
        "--image-height-stddev", "0",
        "--synthetic-input-tokens-mean", str(TEXT_ISL),
        "--synthetic-input-tokens-stddev", "0",
        "--output-tokens-mean", str(OSL),
        "--output-tokens-stddev", "0",
        "--extra-inputs", "ignore_eos:true",
        "--tokenizer", ckpt,
        "--tokenizer-trust-remote-code",
        "--output-artifact-dir", out_dir,
    ]
    with open(f"{out_dir}/aiperf.log", "w") as log:
        subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, check=False)


for variant, ckpt in VARIANTS.items():
    print(f"\n{'='*60}\n>>> {variant}  ({ckpt})\n{'='*60}")
    proc = serve_vllm(ckpt, f"{BENCH_ROOT}/{variant}_vllm.log")
    try:
        if not wait_ready():
            print(f"  !! vLLM did not come up — skipping {variant}."); continue
        for c in CONCURRENCIES:
            out_dir = f"{BENCH_ROOT}/{variant}/c{c}"
            print(f"  -> aiperf concurrency={c}  out={out_dir}")
            run_aiperf(ckpt, c, out_dir)
    finally:
        stop_vllm(proc)
        time.sleep(5)

print(f"\nAll variants done. Artifacts under {BENCH_ROOT}/")

In [ ]:
import os, json, glob
from pathlib import Path

BENCH_ROOT = f"{BASE_PATH}/benchmarks/cosmos-r2-aiperf"
VARIANT_ORDER = ["baseline", "orig_fp8", "distilled_bf16", "distilled_fp8"]


def find_profile_json(out_dir):
    matches = sorted(glob.glob(f"{out_dir}/**/profile_export_aiperf.json", recursive=True))
    return matches[-1] if matches else None


def metric(d, name, stat="avg"):
    m = d.get(name)
    return m.get(stat) if isinstance(m, dict) else m


rows = []
for variant in VARIANT_ORDER:
    var_dir = f"{BENCH_ROOT}/{variant}"
    if not os.path.isdir(var_dir):
        continue
    for c_dir in sorted(Path(var_dir).iterdir(),
                        key=lambda p: int(p.name[1:]) if p.name[1:].isdigit() else 0):
        if not c_dir.name.startswith("c"):
            continue
        c = int(c_dir.name[1:])
        f = find_profile_json(str(c_dir))
        if not f:
            rows.append({"variant": variant, "concurrency": c}); continue
        d = json.load(open(f))
        rows.append({
            "variant":     variant,
            "concurrency": c,
            "ttft":        metric(d, "time_to_first_token"),               # ms
            "e2e":         metric(d, "request_latency"),                   # ms (end-to-end / decision)
            "itl":         metric(d, "inter_token_latency"),               # ms
            "decode_ups":  metric(d, "output_token_throughput_per_user"),  # tok/s per stream
            "tps":         metric(d, "output_token_throughput"),           # tok/s aggregate (secondary)
            "osl":         metric(d, "output_sequence_length"),            # sanity: should be ~uniform (~100)
        })


def fmt(x, p=1):
    return f"{x:9.{p}f}" if isinstance(x, (int, float)) else f"{'  ?  ':>9}"


print(f"{'variant':<16}{'conc':>5}{'TTFT ms':>10}{'e2e ms':>10}{'ITL ms':>9}"
      f"{'decode/usr':>12}{'agg tok/s':>11}{'OSL':>7}")
print("-" * 80)
for r in rows:
    print(f"{r['variant']:<16}{r['concurrency']:>5}{fmt(r.get('ttft'))}{fmt(r.get('e2e'),0)}"
          f"{fmt(r.get('itl'))}{fmt(r.get('decode_ups'))}{fmt(r.get('tps'),0)}{fmt(r.get('osl'))}")

b = next((r for r in rows if r["variant"] == "baseline" and r["concurrency"] == 1), None)
KEYS = ("e2e", "decode_ups", "ttft")
if b and all(b.get(k) for k in KEYS):
    print(f"\nvs BF16 baseline @ concurrency 1  (OSL pinned ~{b.get('osl'):.0f} tokens for all variants):")
    for r in rows:
        if r["concurrency"] == 1 and all(r.get(k) for k in KEYS):
            print(f"  {r['variant']:<16}  end-to-end {b['e2e']/r['e2e']:.2f}x  |  "
                  f"decode {r['decode_ups']/b['decode_ups']:.2f}x  |  TTFT {b['ttft']/r['ttft']:.2f}x")

print(f"\nLoaded {len(rows)} result(s) from {BENCH_ROOT}")

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

COLORS  = {"baseline": "#4C72B0", "orig_fp8": "#DD8452", "distilled_bf16": "#55A868", "distilled_fp8": "#C44E52"}
MARKERS = {"baseline": "o",       "orig_fp8": "s",       "distilled_bf16": "^",       "distilled_fp8": "D"}

by_variant = defaultdict(list)
for r in sorted(rows, key=lambda x: x["concurrency"]):
    by_variant[r["variant"]].append(r)

fig, axes = plt.subplots(1, 4, figsize=(19, 4.3))
fig.suptitle("Single robotics inference — 4-way compression comparison (batch-1 → 8)", fontsize=13)


def plot_metric(ax, key, ylabel, title, lower_better):
    for v in VARIANT_ORDER:
        pts = [(r["concurrency"], r.get(key)) for r in by_variant[v] if r.get(key) is not None]
        if not pts:
            continue
        xs, ys = zip(*pts)
        ax.plot(xs, ys, marker=MARKERS[v], color=COLORS[v], label=v, linewidth=1.8, markersize=6)
    ax.set_xlabel("Concurrency (simultaneous requests)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(sorted({r["concurrency"] for r in rows}))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    ax.annotate("lower = better" if lower_better else "higher = better",
                xy=(0.5, 1.02), xycoords="axes fraction", ha="center", va="bottom",
                fontsize=7, color="gray")


plot_metric(axes[0], "e2e",        "End-to-end latency / decision (ms)", "Latency per decision",   True)
plot_metric(axes[1], "ttft",       "Mean TTFT (ms)",                     "Reaction time (TTFT)",   True)
plot_metric(axes[2], "decode_ups", "Decode tokens / s / stream",         "Per-stream decode rate", False)
plot_metric(axes[3], "tps",        "Aggregate output tokens / s",        "Throughput (secondary)", False)

plt.tight_layout()
plt.savefig(f"{BENCH_ROOT}/benchmarking_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {BENCH_ROOT}/benchmarking_curves.png")

**Reading the curves (single-robot lens):**

- **Latency per decision** (panel 1) This is the end-to-end latency for a request which is dominated by decode but also includes prefill.

- **Per-stream decode rate** (panel 3) Decode is **memory-bandwidth bound**, the per-stream rate falls (ITL rises) under load as the KV cache fills. The compressed model's smaller KV cache means less memory pressure and lower ITL at high request rates.

- **Reaction time / TTFT** (panel 2) TTFT is **prefill**, and for a 768 × 768 image the prefill is anchored by the **BF16 vision tower + `lm_head`, which is deliberately left un-quantized and un-pruned in every variant** (§4). Compression in this notebook touches only the LM, so it can shave just the LM-prefill slice on top of that fixed vision-tower floor.

- **Aggregate throughput** (panel 4, secondary) only matters when one GPU fans out to several cameras / robots.

**Questions to think about:**

1. **Where did the speedup come from?** Compare `orig_fp8` (quantize-only) and `distilled_bf16` (prune-only) against `distilled_fp8`. The decode-rate gains stack — why are FP8 and depth-pruning roughly independent levers, and which one would scale better as the LM tower grows from 2 B toward 7 B?

2. **Why does TTFT move so little?** The vision tower is BF16 in every variant. At what image resolution / output length does decode stop dominating end-to-end latency, so that prefill (and therefore TTFT) would start to matter again?

3. **Bandwidth vs compute.** Per-stream decode is memory-bandwidth bound at batch-1. As concurrency rises, decode shifts toward compute-bound and the GPU batches more work — where do you expect FP8's advantage to grow, and where might it shrink?

4. **Accuracy–latency trade-off.** Cross-reference the §6 recovery table. What accuracy-recovery threshold would you accept for a latency-critical robotics use-case, and below what threshold would you keep the BF16 baseline?

---
## Shutdown

Stop the vLLM server you started in §5.

In [ ]:
!pkill -f 'vllm serve' || true
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

---
## Wrap-up

What you have learned today:

1. **Pruned** Cosmos Reason 2 on the LM tower from 1.7 B → ~1.4 B (15 % cut on the LM; full VLM 2.0 B → ~1.7 B since vision/projector is untouched) by depth-pruning the LM tower from 28 → 24 layers.
2. **Distilled** the pruned student against the unpruned teacher on a Nemotron CC + Math + Code blend — 50 iters on 4×H100 — enough for a smoke-test run; bump to 200-300 iters for full quality recovery.
3. **Quantized** the distilled VLM to FP8 with image-aware calibration; the resulting checkpoint serves directly with vLLM at ~2× the BF16 throughput on H100.
4. **Evaluated** all four variants (baseline / pruned-only / distilled / distilled+FP8) on BLINK (PAI-Bench proxy) and RealWorldQA, and aggregated `% recovered` into a single table.

**Levers if benchmark scores fall short of the recovery target**, in order of expected impact:

1. Bump `--train_iters` 50 → 200-300 for full distillation convergence.
2. Raise `--gbs` 1024 → 2048 if memory allows (halves effective epochs per iter; helps when the corpus is large). Similarly, you can increase the context length.
3. Tilt the corpus blend further toward math: 0.1/0.5/0.4 → 0.1/0.6/0.3.
4. Add an FP8 QAT/QAD pass after §4 (out of scope for this notebook; see `examples/llm_qat/`).